[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tranthithuvan555-wq/TEP/blob/main/Pretrained_Transformers_BERT_GPT.ipynb)

# Máy biến áp được đào tạo trước (BERT, GPT) và đào tạo trên ngôn ngữ đơn giản hóa

Notebook này hướng dẫn:
1. Tại sao **Pre-training** quan trọng và **Transfer Learning** trong NLP
2. Kiến trúc và cách dùng **BERT** (Bidirectional Encoder Representations from Transformers)
3. Kiến trúc và cách dùng **GPT** (Generative Pre-trained Transformer)
4. **Fine-tuning** mô hình trên tập dữ liệu tiếng Việt đơn giản

## 0. Cài đặt thư viện cần thiết

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install transformers torch datasets

In [ ]:
# Import thư viện
import torch
import torch.nn as nn
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForCausalLM,
    pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from torch.utils.data import Dataset, DataLoader

print('Phiên bản PyTorch:', torch.__version__)
print('Thiết bị:', 'GPU' if torch.cuda.is_available() else 'CPU')

---
## 1. Giới thiệu về Pre-trained Transformers

### 1.1 Tại sao Pre-training quan trọng?

**Pre-training** là quá trình huấn luyện mô hình trên tập dữ liệu văn bản **khổng lồ** (hàng tỷ từ) trước khi áp dụng cho nhiệm vụ cụ thể.

Lợi ích:
- Mô hình học được kiến thức ngôn ngữ phổ quát (cú pháp, ngữ nghĩa, kiến thức thế giới)
- **Tiết kiệm tài nguyên**: Không cần dữ liệu có nhãn số lượng lớn cho mỗi tác vụ
- **Kết quả tốt hơn**: Ngay cả với tập dữ liệu downstream nhỏ

### 1.2 Transfer Learning trong NLP

```
Corpus lớn (Wikipedia, sách, web...)
            │
        PRE-TRAINING
   (Học ngôn ngữ nói chung)
            │
      Mô hình Pre-trained
            │
    ┌───────┼───────┐
    │       │       │
Fine-tune Fine-tune Fine-tune
 Phân loại Hỏi đáp  Dịch máy
 cảm xúc    (QA)    ...
```

### 1.3 So sánh BERT và GPT

| Tiêu chí | BERT | GPT |
|---|---|---|
| **Kiến trúc** | Encoder-only | Decoder-only |
| **Hướng đọc** | Hai chiều (bidirectional) | Một chiều (left-to-right) |
| **Mục tiêu pre-train** | Masked Language Model (MLM) + NSP | Causal Language Model (CLM) |
| **Phù hợp cho** | Hiểu văn bản (classification, QA) | Sinh văn bản (generation) |
| **Ví dụ** | BERT, RoBERTa, PhoBERT | GPT-2, GPT-3, GPT-4 |

---
## 2. BERT – Bidirectional Encoder Representations from Transformers

### 2.1 Kiến trúc BERT

BERT sử dụng **Transformer Encoder** hai chiều:

```
[CLS] Con mèo [MASK] trên thảm [SEP]
  │    │    │     │      │    │    │
  └────┴────┴─────┴──────┴────┴────┘
           Bidirectional
        Transformer Encoder × 12
  │    │    │     │      │    │    │
 C_cls C_1  C_2  C_mask  C_4  C_5  C_sep
```

### 2.2 Masked Language Modeling (MLM)

Trong quá trình pre-training, BERT **che ngẫu nhiên 15% các token** và học dự đoán chúng:

> Input:  "Con [MASK] ngồi trên thảm"
> Target: "Con mèo ngồi trên thảm"

### 2.3 Next Sentence Prediction (NSP)

BERT cũng học dự đoán xem câu B có đứng sau câu A trong văn bản gốc không.

In [ ]:
# ============================================================
# Sử dụng PhoBERT – mô hình BERT cho tiếng Việt
# PhoBERT được pre-train trên 20GB văn bản tiếng Việt
# ============================================================

print('Đang tải PhoBERT (mô hình BERT tiếng Việt)...')
print('Lần đầu chạy sẽ tải mô hình (~500MB), vui lòng chờ.\n')

# Tải tokenizer và mô hình
bert_model_name = 'vinai/phobert-base-v2'
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModelForMaskedLM.from_pretrained(bert_model_name)

print(f'Mô hình: {bert_model_name}')
print(f'Kích thước từ điển: {bert_tokenizer.vocab_size:,} tokens')
total_params = sum(p.numel() for p in bert_model.parameters())
print(f'Tổng số tham số: {total_params:,} (~{total_params/1e6:.0f}M)')

In [ ]:
# ============================================================
# Demo: Fill-Mask với câu tiếng Việt
# Dự đoán từ bị che ([MASK]) trong câu
# ============================================================

# Tạo pipeline fill-mask
fill_mask = pipeline(
    'fill-mask',
    model=bert_model,
    tokenizer=bert_tokenizer,
    top_k=5  # Trả về 5 gợi ý tốt nhất
)

# Các câu tiếng Việt với token bị che
sentences = [
    'Hà Nội là <mask> của Việt Nam.',
    'Con mèo đang <mask> trên ghế.',
    'Học sinh chăm chỉ sẽ đạt được <mask> tốt.',
]

print('=== Demo Fill-Mask với PhoBERT ===')
for sentence in sentences:
    print(f'\nCâu gốc: {sentence}')
    print('Dự đoán top-5:')
    results = fill_mask(sentence)
    for i, result in enumerate(results, 1):
        score = result['score']
        token = result['token_str']
        filled = result['sequence']
        print(f'  {i}. "{token}" (xác suất: {score:.4f}) → {filled}')

In [ ]:
# ============================================================
# Trực quan hóa: Tokenization của PhoBERT
# PhoBERT dùng BPE (Byte-Pair Encoding) để tách từ
# ============================================================

sentence = 'Tôi yêu học máy và xử lý ngôn ngữ tự nhiên'
tokens = bert_tokenizer.tokenize(sentence)
token_ids = bert_tokenizer.encode(sentence)

print('=== Tokenization với PhoBERT ===')
print(f'Câu gốc:  {sentence}')
print(f'Tokens:   {tokens}')
print(f'Token IDs: {token_ids}')
print(f'Số token: {len(tokens)}')

---
## 3. GPT – Generative Pre-trained Transformer

### 3.1 Kiến trúc GPT (Autoregressive)

GPT sử dụng **Transformer Decoder** với causal (masked) attention:

```
Token:  Tôi   yêu   học  máy
        │      │      │     │
  ┌─────┴──────┴──────┴─────┘
  │   Causal (masked) attention:
  │   "Tôi" chỉ nhìn thấy "Tôi"
  │   "yêu" nhìn thấy "Tôi", "yêu"
  │   "học" nhìn thấy "Tôi", "yêu", "học"
  └────────────────────────────
        │      │      │     │
       yêu    học    máy    ?
  (Dự đoán token tiếp theo tại mỗi vị trí)
```

### 3.2 Mục tiêu Causal Language Modeling (CLM)

GPT học dự đoán xác suất của token tiếp theo:

$$P(x_t | x_1, x_2, \ldots, x_{t-1})$$

Trong quá trình sinh văn bản, GPT sinh từng token một cách tuần tự.

In [ ]:
# ============================================================
# Sử dụng GPT-2 từ HuggingFace
# ============================================================

print('Đang tải mô hình GPT-2...')

# Tải tokenizer và mô hình GPT-2
gpt_model_name = 'gpt2'
gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)
gpt_model = AutoModelForCausalLM.from_pretrained(gpt_model_name)

# GPT-2 không có padding token; dùng eos_token thay thế
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

total_params = sum(p.numel() for p in gpt_model.parameters())
print(f'Mô hình: {gpt_model_name}')
print(f'Kích thước từ điển: {gpt_tokenizer.vocab_size:,} tokens')
print(f'Tổng số tham số: {total_params:,} (~{total_params/1e6:.0f}M)')

In [ ]:
# ============================================================
# Demo: Text Generation với GPT-2
# Sinh văn bản tiếp theo từ một đoạn văn bản đầu vào
# ============================================================

# Tạo pipeline sinh văn bản
text_gen = pipeline(
    'text-generation',
    model=gpt_model,
    tokenizer=gpt_tokenizer,
)

# Prompt tiếng Anh (GPT-2 gốc được train chủ yếu trên tiếng Anh)
prompts = [
    'Artificial intelligence is transforming',
    'The future of natural language processing',
]

print('=== Demo Text Generation với GPT-2 ===')
for prompt in prompts:
    print(f'\nPrompt: "{prompt}"')
    results = text_gen(
        prompt,
        max_new_tokens=40,    # Sinh thêm 40 token
        num_return_sequences=2,  # Sinh 2 câu khác nhau
        temperature=0.9,      # Điều chỉnh độ sáng tạo
        do_sample=True,
        truncation=True,
    )
    for i, result in enumerate(results, 1):
        print(f'  Kết quả {i}: {result["generated_text"]}')

---
## 4. Fine-tuning trên tập dữ liệu tiếng Việt

### Quy trình Fine-tuning

```
Mô hình Pre-trained (BERT/GPT)
           │
    Tải trọng số
           │
  Tập dữ liệu tiếng Việt
    (có thể có nhãn)
           │
    Fine-tuning
  (learning rate nhỏ)
           │
  Mô hình đã Fine-tune
   (hoạt động tốt trên
    tiếng Việt)
```

Trong phần này, chúng ta sẽ fine-tune GPT-2 trên một tập văn bản tiếng Việt đơn giản để nó có thể sinh văn bản phong cách tiếng Việt.

In [ ]:
# ============================================================
# Tập dữ liệu tiếng Việt đơn giản để fine-tune
# ============================================================

# Tập văn bản tiếng Việt đơn giản (thay bằng tập dữ liệu thực tế để có kết quả tốt hơn)
vietnamese_texts = [
    'Việt Nam là một đất nước tươi đẹp với lịch sử hào hùng.',
    'Người Việt Nam nổi tiếng với lòng hiếu khách và sự thân thiện.',
    'Hà Nội là thủ đô ngàn năm văn hiến của Việt Nam.',
    'Thành phố Hồ Chí Minh là trung tâm kinh tế lớn nhất cả nước.',
    'Vịnh Hạ Long là một trong những kỳ quan thiên nhiên của thế giới.',
    'Ẩm thực Việt Nam phong phú và đa dạng, nổi tiếng khắp thế giới.',
    'Pho là món ăn truyền thống nổi tiếng nhất của Việt Nam.',
    'Áo dài là trang phục truyền thống và biểu tượng văn hóa Việt.',
    'Ngôn ngữ tiếng Việt có sáu thanh điệu khác nhau.',
    'Đất nước Việt Nam có hơn 54 dân tộc anh em cùng chung sống.',
    'Sông Mê Kông chảy qua đồng bằng phía Nam mang phù sa màu mỡ.',
    'Học sinh Việt Nam nổi tiếng chăm chỉ và đạt nhiều thành tích học tập cao.',
    'Công nghệ thông tin đang phát triển mạnh mẽ tại Việt Nam.',
    'Giáo dục là quốc sách hàng đầu của nước ta.',
    'Thiên nhiên Việt Nam rất đa dạng từ núi rừng đến biển cả.',
]

print(f'Số lượng câu trong tập dữ liệu: {len(vietnamese_texts)}')
print('\nMột số ví dụ:')
for text in vietnamese_texts[:3]:
    print(f'  - {text}')

In [ ]:
# ============================================================
# Tạo Dataset cho fine-tuning
# ============================================================

class VietnameseTextDataset(Dataset):
    """Dataset tiếng Việt cho fine-tuning mô hình ngôn ngữ."""
    
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Mã hóa tất cả văn bản
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding='max_length',
            return_tensors='pt'
        )
    
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):
        # Trả về input_ids và labels (labels = input_ids cho CLM)
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['input_ids'][idx].clone(),
        }
        return item


# Tạo dataset
dataset = VietnameseTextDataset(vietnamese_texts, gpt_tokenizer, max_length=64)
print(f'Kích thước dataset: {len(dataset)} mẫu')
print(f'Kích thước batch đầu tiên: {dataset[0]["input_ids"].shape}')

# Kiểm tra tokenization
sample = dataset[0]
decoded = gpt_tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
print(f'\nVăn bản gốc: {vietnamese_texts[0]}')
print(f'Sau tokenization và decode: {decoded[:80]}...')

In [ ]:
# ============================================================
# Fine-tuning GPT-2 trên tập dữ liệu tiếng Việt
# Chỉ huấn luyện 3 epoch để minh họa (tập dữ liệu rất nhỏ)
# ============================================================

import os

# Cấu hình huấn luyện
training_args = TrainingArguments(
    output_dir='/tmp/gpt2-vietnamese',   # Thư mục lưu mô hình
    overwrite_output_dir=True,
    num_train_epochs=3,                  # Số epoch huấn luyện
    per_device_train_batch_size=4,       # Kích thước batch
    save_steps=50,                       # Lưu mô hình mỗi 50 bước
    save_total_limit=2,                  # Giữ tối đa 2 checkpoint
    logging_steps=10,                    # In log mỗi 10 bước
    learning_rate=5e-5,                  # Tốc độ học
    warmup_steps=10,                     # Số bước warm-up
    no_cuda=not torch.cuda.is_available(),
    report_to='none',                    # Không báo cáo lên wandb, etc.
)

# Khởi tạo Trainer
trainer = Trainer(
    model=gpt_model,
    args=training_args,
    train_dataset=dataset,
)

print('Bắt đầu fine-tuning GPT-2 trên dữ liệu tiếng Việt...')
print(f'Tập dữ liệu: {len(dataset)} câu')
print(f'Số epoch: 3')
print(f'Batch size: 4')
print()

# Chạy fine-tuning
trainer.train()
print('\nFine-tuning hoàn tất!')

In [ ]:
# ============================================================
# Đánh giá kết quả fine-tuning
# So sánh trước và sau khi fine-tune
# ============================================================

# Tính perplexity trên tập dữ liệu (chỉ số đánh giá mô hình ngôn ngữ)
import math

def calculate_perplexity(model, tokenizer, texts):
    """Tính perplexity – chỉ số đánh giá mô hình ngôn ngữ.
    Perplexity càng thấp, mô hình dự đoán văn bản càng tốt.
    """
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for text in texts:
            # Mã hóa văn bản
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
            input_ids = inputs['input_ids']
            
            if input_ids.size(1) < 2:
                continue
            
            # Tính loss
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss.item()
            n_tokens = input_ids.size(1)
            
            total_loss += loss * n_tokens
            total_tokens += n_tokens
    
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    perplexity = math.exp(avg_loss)
    return perplexity


# Tính perplexity sau fine-tuning
ppl = calculate_perplexity(gpt_model, gpt_tokenizer, vietnamese_texts)
print(f'=== Đánh giá sau Fine-tuning ===')
print(f'Perplexity trên tập tiếng Việt: {ppl:.2f}')
print()
print('Giải thích perplexity:')
print('  - Perplexity = 1: Mô hình dự đoán hoàn hảo')
print('  - Perplexity = N: Mô hình "bối rối" như chọn ngẫu nhiên trong N lựa chọn')
print('  - Perplexity càng thấp càng tốt')

In [ ]:
# ============================================================
# Sinh văn bản sau khi fine-tune
# ============================================================

def generate_vietnamese(model, tokenizer, prompt, max_new_tokens=50):
    """Sinh văn bản tiếng Việt từ một đoạn prompt."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt')
    
    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            do_sample=True,
            top_k=20,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print('=== Sinh văn bản sau Fine-tuning ===')
test_prompts = [
    'Việt Nam là',
    'Người Việt Nam',
    'Hà Nội',
]

for prompt in test_prompts:
    result = generate_vietnamese(gpt_model, gpt_tokenizer, prompt)
    print(f'Prompt: "{prompt}"')
    print(f'Output: {result}')
    print()

print('Lưu ý: Với tập dữ liệu nhỏ (15 câu), mô hình chỉ học được ít.')
print('Để có kết quả tốt hơn, cần tập dữ liệu tiếng Việt lớn hơn nhiều.')

---
## 5. Tổng kết

### Những điểm quan trọng:

| Mô hình | Kiến trúc | Tác vụ phù hợp | Ví dụ tiếng Việt |
|---|---|---|---|
| **BERT** | Encoder-only, bidirectional | Phân loại, NER, QA | PhoBERT, viBERT |
| **GPT-2** | Decoder-only, causal | Sinh văn bản | Fine-tune trên corpus tiếng Việt |
| **T5/mT5** | Encoder-Decoder | Dịch máy, tóm tắt | mT5 (đa ngôn ngữ) |

### Mẹo fine-tuning hiệu quả:
1. **Dữ liệu nhiều và sạch**: Chất lượng dữ liệu quan trọng hơn số lượng
2. **Learning rate nhỏ**: Thường dùng 1e-5 đến 5e-5 để không "quên" kiến thức cũ
3. **Warm-up steps**: Giúp ổn định huấn luyện ban đầu
4. **Đánh giá thường xuyên**: Theo dõi loss trên tập validation
5. **Early stopping**: Dừng khi validation loss không cải thiện

### Tài liệu tham khảo:
- [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)
- [PhoBERT: Pre-trained language models for Vietnamese](https://arxiv.org/abs/2003.00744)
- [HuggingFace Transformers Documentation](https://huggingface.co/docs/transformers)
- [Fine-tuning a pretrained model (HuggingFace Tutorial)](https://huggingface.co/docs/transformers/training)